# EXTRACCIÓN AUTOMÁTICA DE DATOS DE ARTÍCULOS CIENTÍFICOS CON UN LLM

Aitor Madoz González. Proyecto del Máster en Big Data Science e Inteligencia Artificial (Universidad de Navarra), asignatura de Técnicas de Recogida de Datos.

En este proyecto partimos de seis artículos científicos en PDF y construimos una tabla con una fila por artículo y una columna por cada dato de interés, dejando un valor nulo cuando el dato no aparece. Las columnas finales son el título, el problema de salud que estudia el artículo, la población estudiada, las palabras clave y el número de referencias bibliográficas.

Primero leemos el texto de cada PDF con la librería pypdf. El número de referencias lo calculamos directamente con Python, buscando dónde empieza la bibliografía al final de cada artículo y contando las referencias según su formato, numerado o de autor y año. El resto de datos se los pedimos a un modelo de lenguaje descargado de Hugging Face, Qwen2.5-3B-Instruct, que ejecutamos en la GPU de Google Colab. Al modelo le pasamos el texto completo de cada artículo con un prompt que le pide responder en formato JSON y poner null cuando un dato no aparece. Cada artículo se convierte en un registro que se añade con append a una lista, y al final esa lista se transforma en un dataframe, que mostramos también en una tabla con mejor presentación.

En el camino aparecieron varios problemas. Al principio intentamos ejecutar el modelo en local en un Mac: con la GPU del Mac se quedaba bloqueado al generar la respuesta y con MLX, la librería de Apple, era demasiado lento, así que pasamos a Colab. El número de referencias cambiaba según la versión de pypdf, porque cada versión extrae el texto de forma algo distinta, por lo que fijamos la versión para que el resultado sea siempre el mismo. En uno de los PDF los números aparecían como nombres de glifos en lugar de dígitos, debido a la fuente del documento, y los convertimos a números antes de procesar el texto. El modelo, además, cambiaba los nombres de las claves del JSON cuando se las pedíamos en español, así que las dejamos en inglés. Por último, en una primera versión le pedíamos datos que hay que interpretar leyendo todo el estudio, como el número de participantes, el uso de placebo, el número de grupos o el tipo de estudio, y el resultado fue muy malo: dejaba casi todo vacío o confundía conceptos. Por eso cambiamos a datos que se reconocen con claridad en el texto.

El resultado es una tabla completa para los seis artículos. La principal limitación es que, al tratarse de un modelo pequeño, funciona bien con datos que aparecen claramente en el texto pero falla con los que requieren interpretación. El objetivo era automatizar la extracción; los datos obtenidos requerirían una revisión humana antes de usarse. Los artículos no se incluyen en el repositorio: son de acceso abierto y se pueden descargar desde los enlaces del README.

### INSTALAMOS LAS LIBRERÍAS

In [1]:
%pip install -q pypdf==6.21.2 transformers accelerate pandas great_tables

### SUBIMOS LOS PDF A COLAB

In [2]:
import os
import shutil
from google.colab import files

CARPETA_PDFS = "pdfs"

os.makedirs(CARPETA_PDFS, exist_ok=True)
subidos = files.upload()
for nombre in subidos:
    shutil.move(nombre, os.path.join(CARPETA_PDFS, nombre))

Saving fpsyg-08-00308.pdf to fpsyg-08-00308.pdf
Saving fpsyg-09-01240.pdf to fpsyg-09-01240.pdf
Saving fpsyg-11-01354.pdf to fpsyg-11-01354.pdf
Saving fpsyg-12-639236.pdf to fpsyg-12-639236.pdf
Saving fpsyt-14-1301143.pdf to fpsyt-14-1301143.pdf
Saving fpubh-10-1045777.pdf to fpubh-10-1045777.pdf


### IMPORTAMOS LAS LIBRERÍAS Y DEFINIMOS LOS PARÁMETROS

In [3]:
import re
import json
import logging

import pandas as pd
import torch
from pypdf import PdfReader
from transformers import AutoTokenizer, AutoModelForCausalLM
from great_tables import GT

MODELO = "Qwen/Qwen2.5-3B-Instruct"
MAX_TOKENS_RESPUESTA = 300

pdfs = sorted(f for f in os.listdir(CARPETA_PDFS) if f.endswith(".pdf"))
print(len(pdfs))
for pdf in pdfs:
    print(pdf)

6
fpsyg-08-00308.pdf
fpsyg-09-01240.pdf
fpsyg-11-01354.pdf
fpsyg-12-639236.pdf
fpsyt-14-1301143.pdf
fpubh-10-1045777.pdf


### LEEMOS LOS PDF Y CONTAMOS LAS REFERENCIAS

In [4]:
logging.getLogger("pypdf").setLevel(logging.ERROR)

DIGITOS = {"zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
           "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9"}


def limpiar_texto(texto):
    return re.sub(r"/(zero|one|two|three|four|five|six|seven|eight|nine)\.tnum",
                  lambda m: DIGITOS[m.group(1)], texto)


def leer_pdf(ruta):
    lector = PdfReader(ruta)
    return "\n".join(limpiar_texto(pagina.extract_text() or "") for pagina in lector.pages)


def contar_referencias(texto):
    inicio = list(re.finditer(r"\n\s*(REFERENCES|References)\s*\n", texto))
    if not inicio:
        return None
    seccion = texto[inicio[-1].end():]

    numeros = [int(n) for n in re.findall(r"^\s*(\d{1,3})\.\s+[A-Z]", seccion, flags=re.M)]
    if len(numeros) > 5:
        return max(numeros)
    return len(re.findall(r"\(\s*(?:19|20)\d{2}[a-z]?\)", seccion))


documentos = []
for pdf in pdfs:
    texto = leer_pdf(os.path.join(CARPETA_PDFS, pdf))
    documentos.append({
        "archivo": pdf,
        "referencias": contar_referencias(texto),
        "texto": texto,
    })

pd.DataFrame(documentos).drop(columns="texto")

,archivo,referencias
0,fpsyg-08-00308.pdf,58
1,fpsyg-09-01240.pdf,108
2,fpsyg-11-01354.pdf,99
3,fpsyg-12-639236.pdf,72
4,fpsyt-14-1301143.pdf,78
5,fpubh-10-1045777.pdf,22


### DESCARGAMOS Y CARGAMOS EL MODELO DE HUGGING FACE

In [5]:
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float16).to(dispositivo)

print("Modelo cargado en:", dispositivo)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo cargado en: cuda


### PROBAMOS EL PROMPT CON UN ARTÍCULO

In [6]:
PROMPT = """You are extracting metadata from a scientific article.
Answer ONLY with a JSON object with these keys:
- "title": the title of the article
- "health_condition": the main disease, disorder or health problem the article is about, in a few words (for example "depression", "chronic pain")
- "population": who the people studied are, in a few words (for example "healthy adults", "adolescents"), or null if the article does not study people
- "keywords": the keywords listed after "Keywords", as a single string separated by commas
If a value does not appear in the text, use null. Do not add any explanation.

Text:
{texto}"""


def preguntar_al_llm(texto):
    mensajes = [{"role": "user", "content": PROMPT.format(texto=texto)}]
    prompt_chat = tokenizer.apply_chat_template(mensajes, add_generation_prompt=True, tokenize=False)
    entrada = tokenizer(prompt_chat, return_tensors="pt").to(dispositivo)

    salida = modelo.generate(**entrada, max_new_tokens=MAX_TOKENS_RESPUESTA, do_sample=False)
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    return tokenizer.decode(tokens_nuevos, skip_special_tokens=True)


print(preguntar_al_llm(documentos[0]["texto"]))

```json
{
  "title": "Placebo and Nocebo Effects: The Advantage of Measuring Expectations and Psychological Factors",
  "health_condition": "pain",
  "population": "healthy adults",
  "keywords": "acute pain, anxiety, conditioning, expectation, fear, neuroticism, suggestibility"
}
```


### EXTRAEMOS LOS DATOS DE TODOS LOS ARTÍCULOS Y MONTAMOS EL DATAFRAME

In [7]:
def extraer_json(respuesta):
    encontrado = re.search(r"\{.*\}", respuesta, flags=re.S)
    if not encontrado:
        return {}
    try:
        return json.loads(encontrado.group())
    except json.JSONDecodeError:
        return {}


filas = []
for documento in documentos:
    print("Procesando", documento["archivo"], "...")
    datos = extraer_json(preguntar_al_llm(documento["texto"]))
    filas.append({
        "archivo": documento["archivo"],
        "titulo": datos.get("title"),
        "problema_salud": datos.get("health_condition"),
        "poblacion": datos.get("population"),
        "palabras_clave": datos.get("keywords"),
        "referencias": documento["referencias"],
    })

tabla = pd.DataFrame(filas)
tabla

Procesando fpsyg-08-00308.pdf ...
Procesando fpsyg-09-01240.pdf ...
Procesando fpsyg-11-01354.pdf ...
Procesando fpsyg-12-639236.pdf ...
Procesando fpsyt-14-1301143.pdf ...
Procesando fpubh-10-1045777.pdf ...


,archivo,titulo,problema_salud,poblacion,palabras_clave,referencias
0,fpsyg-08-00308.pdf,Placebo and Nocebo Effects: The Advantage of M...,pain,healthy adults,"acute pain, anxiety, conditioning, expectation...",58
1,fpsyg-09-01240.pdf,Application of Bayes’ Theorem in Valuating Dep...,depression,"healthy adults, adolescents, patients with psy...","Bayes’ theorem, diagnostic accuracy, depressio...",108
2,fpsyg-11-01354.pdf,Putting the 'Art' Into the 'Art of Medicine': ...,"chronic pain, anxiety, depression, migraine, m...","patients, healthcare providers","placebo, placebo effects, expectancy, mindset,...",99
3,fpsyg-12-639236.pdf,"Placebo Effects on Stress, but Not on Pain",stress,None,"placebo effect, placebo response, experimenter...",72
4,fpsyt-14-1301143.pdf,Placebo stimulates neuroplasticity in depressi...,depression,patients with depression,"neuroplasticity, placebo, depression, mechanis...",78
5,fpubh-10-1045777.pdf,Cognitive distortion based explainable depress...,depression,adolescents,"depression detection, cognitive distortion, co...",22


### MOSTRAMOS LA TABLA FINAL

In [8]:
(
    GT(tabla)
    .tab_header(
        title="Metadatos de los estudios",
        subtitle="Extraídos automáticamente de los PDF con Qwen2.5-3B-Instruct",
    )
    .cols_label(
        archivo="Archivo",
        titulo="Título",
        problema_salud="Problema de salud",
        poblacion="Población estudiada",
        palabras_clave="Palabras clave",
        referencias="Referencias",
    )
    .sub_missing(missing_text="—")
    .data_color(columns="referencias", palette=["#e8f1fb", "#2a6fbb"])
    .opt_stylize(style=1, color="blue")
)

GT(_tbl_data=                archivo                                             titulo  \
0    fpsyg-08-00308.pdf  Placebo and Nocebo Effects: The Advantage of M...   
1    fpsyg-09-01240.pdf  Application of Bayes’ Theorem in Valuating Dep...   
2    fpsyg-11-01354.pdf  Putting the 'Art' Into the 'Art of Medicine': ...   
3   fpsyg-12-639236.pdf         Placebo Effects on Stress, but Not on Pain   
4  fpsyt-14-1301143.pdf  Placebo stimulates neuroplasticity in depressi...   
5  fpubh-10-1045777.pdf  Cognitive distortion based explainable depress...   

                                      problema_salud  \
0                                               pain   
1                                         depression   
2  chronic pain, anxiety, depression, migraine, m...   
3                                             stress   
4                                         depression   
5                                         depression   

                                           poblacion  \
0                                     healthy adults   
1  healthy adults, adolescents, patients with psy...   
2                     patients, healthcare providers   
3                                               None   
4                           patients with depression   
5                                        adolescents   

                                      palabras_clave  referencias  
0  acute pain, anxiety, conditioning, expectation...           58  
1  Bayes’ theorem, diagnostic accuracy, depressio...          108  
2  placebo, placebo effects, expectancy, mindset,...           99  
3  placebo effect, placebo response, experimenter...           72  
4  neuroplasticity, placebo, depression, mechanis...           78  
5  depression detection, cognitive distortion, co...           22  , _body=<great_tables._gt_data.Body object at 0x7d83d5d552b0>, _boxhead=Boxhead([ColInfo(var='archivo', type=<ColInfoTypeEnum.default: 1>, column_label='Archivo', column_align='left', column_width=None, stub_level=None), ColInfo(var='titulo', type=<ColInfoTypeEnum.default: 1>, column_label='Título', column_align='left', column_width=None, stub_level=None), ColInfo(var='problema_salud', type=<ColInfoTypeEnum.default: 1>, column_label='Problema de salud', column_align='left', column_width=None, stub_level=None), ColInfo(var='poblacion', type=<ColInfoTypeEnum.default: 1>, column_label='Población estudiada', column_align='left', column_width=None, stub_level=None), ColInfo(var='palabras_clave', type=<ColInfoTypeEnum.default: 1>, column_label='Palabras clave', column_align='left', column_width=None, stub_level=None), ColInfo(var='referencias', type=<ColInfoTypeEnum.default: 1>, column_label='Referencias', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x7d83d5d56660>, _spanners=Spanners([]), _heading=Heading(title='Metadatos de los estudios', subtitle='Extraídos automáticamente de los PDF con Qwen2.5-3B-Instruct', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7d83d5d55550>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7d840c24f750>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns='referencias', rows=[0], mask=None), grpname=None, colname='referencias', rownum=0, colnum=None, styles=[CellStyleText(color='#000000', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None), CellStyleFill(color='#a2b8e0')]), StyleInfo(locname=LocBody(columns='referencias', rows=[1], mask=None), grpname=None, colname='referencias', rownum=1, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None), CellStyleFill(color='#2a6fbb')]), StyleInfo(locname=LocBody(columns='referencias', rows=[2], mask=None), grpname=No